# My Chapter 2 Worklog

Personal notes, experiments, and observations for the chapter exercises.

## Exercise 2.1 - Tokenizing unseen words


In [ ]:
from pathlib import Path
import subprocess
import sys

# Make local package importable regardless notebook working directory.
cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate repo root (missing pyproject.toml)")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Tokenizer
except ModuleNotFoundError:
    print("Installing local package into current kernel environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(repo_root)])
    from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Tokenizer

# I picked one invented token and one non-English phrase.
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

tok_file = Path("qwen3") / "tokenizer-base.json"
tok = Qwen3Tokenizer(tokenizer_file_path=tok_file)

sample_text = "Hi, blorvathionz! Bonjour et merci."
ids = tok.encode(sample_text)

print("Token-by-token decode:")
for token_id in ids:
    print(f"{[token_id]} -> {tok.decode([token_id])}")

Installing local package into current kernel environment...


## Exercise 2.2 - Compare CPU vs accelerator runs

In [ ]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream,
    generate_text_basic_stream_cache,
    generate_stats,
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B,
)

# Keep auto-selected device (CUDA/MPS/CPU depending on machine).
run_device = get_device()

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

tok_path = Path("qwen3") / "tokenizer-base.json"
weights_path = Path("qwen3") / "qwen3-0.6B-base.pth"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tok_path)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(weights_path))

model.to(run_device);

In [ ]:
prompt = "In one sentence, explain why transformers scale well."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=run_device,
).unsqueeze(0)

In [ ]:
import time

max_new_tokens = 100
t0 = time.time()
generated_token_chunks = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_ids,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    token_as_list = token.squeeze(0).tolist()
    print(tokenizer.decode(token_as_list), end="", flush=True)
    generated_token_chunks.append(token.squeeze(0))

t1 = time.time()
out_ids = torch.cat(generated_token_chunks, dim=0)
generate_stats(out_ids, tokenizer, t0, t1)

In [ ]:
t0 = time.time()
generated_token_chunks = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    token_as_list = token.squeeze(0).tolist()
    print(tokenizer.decode(token_as_list), end="", flush=True)
    generated_token_chunks.append(token.squeeze(0))

t1 = time.time()
out_ids = torch.cat(generated_token_chunks, dim=0)
generate_stats(out_ids, tokenizer, t0, t1)

In [ ]:
if run_device.type == "mps":
    print(
        f"`torch.compile` is not supported for {model.__class__.__name__} on MPS right now."
    )
    compiled_model = model
else:
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 8):
        torch._dynamo.config.allow_unspec_int_on_nn_module = True

    compiled_model = torch.compile(model)

In [ ]:
for run_idx in range(3):

    t0 = time.time()
    generated_token_chunks = []

    for token in generate_text_basic_stream(
        model=compiled_model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        token_as_list = token.squeeze(0).tolist()
        print(tokenizer.decode(token_as_list), end="", flush=True)
        generated_token_chunks.append(token.squeeze(0))

    t1 = time.time()

    if run_idx == 0:
        print("Warm-up")
    else:
        print(f"Measured run {run_idx}")

    out_ids = torch.cat(generated_token_chunks, dim=0)
    generate_stats(out_ids, tokenizer, t0, t1)

    print(f"\n{30 * '-'}\n")

In [ ]:
for run_idx in range(3):

    t0 = time.time()
    generated_token_chunks = []

    for token in generate_text_basic_stream_cache(
        model=compiled_model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        token_as_list = token.squeeze(0).tolist()
        print(tokenizer.decode(token_as_list), end="", flush=True)
        generated_token_chunks.append(token.squeeze(0))

    t1 = time.time()

    if run_idx == 0:
        print("Warm-up")
    else:
        print(f"Measured run {run_idx}")

    out_ids = torch.cat(generated_token_chunks, dim=0)
    generate_stats(out_ids, tokenizer, t0, t1)

    print(f"\n{30 * '-'}\n")

## Exercise 3.1 - Add stronger checker tests

In [ ]:
from reasoning_from_scratch.ch03 import run_demos_table

stress_tests = [
    # Equivalent fraction formats
    ("mine_01", "4/8", "1/2", True),

    # Same value in decimal and percent
    ("mine_02", "0.25", "25%", True),

    # Different notation for powers
    ("mine_03", "2^5", "32", True),

    # Unicode minus vs normal minus
    ("mine_04", "−7", "-7", True),
]

run_demos_table(stress_tests)

In [ ]:
edge_case_raw = [
    ("mine_05", "The final number I got is 3.", "3", True)
]

run_demos_table(edge_case_raw)

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate

edge_case_extracted = [
    (
        "mine_06",
        extract_final_candidate("The final number I got is 3."),
        "3",
        True,
    )
]
run_demos_table(edge_case_extracted)

## Exercise 3.2 - Compute average response length from report files

In [ ]:
import json
from pathlib import Path

MODEL_KIND = "base"

device_tag = "mps"  # change to "cpu" or "cuda" if needed
report_file = Path(f"math500-{device_tag}.jsonl")
if not report_file.exists():
    raise FileNotFoundError(
        f"{report_file} not found. Generate it first from the chapter 3 notebook."
    )

records = []
with open(report_file, "r", encoding="utf-8") as f:
    for raw_line in f:
        if raw_line.strip():
            records.append(json.loads(raw_line))

print("Loaded rows:", len(records))

In [ ]:
print(records[0].keys())

In [ ]:
from reasoning_from_scratch.qwen3 import download_qwen3_small, Qwen3Tokenizer

if MODEL_KIND == "base":
    download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif MODEL_KIND == "reasoning":
    download_qwen3_small(kind="reasoning", tokenizer_only=True, out_dir="qwen3")
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError("MODEL_KIND must be 'base' or 'reasoning'.")

In [ ]:
token_total = 0

for row in records:
    token_count = len(tokenizer.encode(row["generated_text"]))
    token_total += token_count

avg_tokens = token_total / len(records)
print(f"Average generated length: {avg_tokens:.2f} tokens")

## Exercise 3.3 - Evaluate on a larger dataset slice

In [ ]:
# Instead of the default tiny subset, evaluate a larger slice.
expanded_subset = math_data[:100]  # can increase this further if runtime allows

# Reuse whichever device variable is available in the notebook session.
eval_device = device if "device" in globals() else run_device

num_correct, num_examples, acc = evaluate_math500_stream(
    model,
    tokenizer,
    eval_device,
    math_data=expanded_subset,
    max_new_tokens=2048,
    verbose=False,
)

print(f"Evaluated {num_examples} examples, accuracy={acc:.3f}")